### IMPLEMENTING A SIMPLIFIED ATTENTION MECHANISM:

In [1]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [2]:
query = inputs[1]  # 2nd input token is the query

attn_scores = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores[i] = torch.dot(x_i, query) # dot product (transpose not necessary here since they are 1-dim vectors)

print(attn_scores)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


We normalize each of the attention scores that we computed previously.

The main goal behind the normalization is to obtain attention weights that sum up to 1.

This normalization is a convention that is useful for interpretation and for maintaining training stability in an LLM.

Here's a straightforward method for achieving this normalization step:


In [3]:
attn_weights_tmp = attn_scores / attn_scores.sum()

print("Attention weights:", attn_weights_tmp)
print("Sum:", attn_weights_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


In practice, it's more common and advisable to use the softmax function for normalization.

This approach is better at managing extreme values and offers more favorable gradient properties during training.

Below is a basic implementation of the softmax function for normalizing the attention scores: 

In [5]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_naive = softmax_naive(attn_scores)

print("Attention weights:", attn_weights_naive)
print("Sum:", attn_weights_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


As the output shows, the softmax function also meets the objective and normalizes the attention weights such that they sum to 1:

In addition, the softmax function ensures that the attention weights are always positive. This makes the output interpretable as probabilities or relative importance, where higher weights indicate greater importance.

Note that this naive softmax implementation (softmax_naive) may encounter numerical instability problems, such as overflow and underflow, when dealing with large or small input values.

Therefore, in practice, it's advisable to use the PyTorch implementation of softmax, which has been extensively optimized for performance:

pytorch version uses e^x1 - max / sum, e^x2 - max / sum, ... instead of normal e^x1/sum like implemented above for stability and performance.


In [6]:
attn_weights = torch.softmax(attn_scores, dim=0)
print("Attention weights:", attn_weights)
print("Sum:", attn_weights.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


The context vector z(2) is calculated as a weighted sum of all input vectors.

This involves multiplying each input vector by its corresponding attention weight:

In [7]:
query = inputs[1] # 2nd input token is the query

context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights[i] * x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


We can extend this computation to calculate attention weights and context vectors for all inputs.

First, we add an additional for-loop to compute the dot products for all pairs of inputs.

In [8]:
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


When computing the preceding attention score tensor, we used for-loops in Python.

However, for-loops are generally slow, and we can achieve the same results using matrix multiplication:

In [9]:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


We now normalize each row so that the values in each row sum to 1:

By setting dim=-1, we are instructing the softmax function to apply the normalization along the last dimension of the attn_scores tensor.

If attn_scores is a 2D tensor (for example, with a shape of [rows, columns]), dim=-1 will normalize across the columns so that the values in each row (summing over the column dimension) sum up to 1.

In [10]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


We now use these attention weights to compute all context vectors via matrix multiplication:

In [11]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


### 🧠 Self-Attention with Trainable Weights: Conceptual Summary

Self-attention transforms input embeddings into context-aware representations by learned projections.

#### 1. Linear Projections (The QKV Framework)
We transform each input vector ($x$) into three functional roles using trainable weight matrices ($W_q, W_k, W_v$):
* **Query ($q$):** "What I am looking for."
* **Key ($k$):** "What I offer/how to index me."
* **Value ($v$):** "The actual content I contribute."
> **Note:** We often project from an input dimension ($d_{in}$) to a different output dimension ($d_{out}$). While ($d_{in}$)​ and ($d_{out}$)​ are often the same in models like GPT, the mechanism allows you to project the input into a different dimensional space (e.g., from 3D to 2D) using the weight matrices.

#### 2. Computing Attention Scores
We calculate the "relevance" of every element to our current query by taking the **dot product** between the Query ($q$) and all Keys ($k$).


#### 3. Scaling & Normalization
* **Scaling:** We divide scores by $\sqrt{d_k}$ (square root of the key dimension) to prevent extreme values that would "kill" gradients during training.
* **Softmax:** We apply Softmax to turn these scaled scores into **weights** that sum to 1. This determines the percentage of "attention" paid to each input.

#### 4. The Context Vector ($z$)
The final output for a token is a **weighted sum** of all Value ($v$) vectors. 
* If the attention weight for a specific token is high, its Value vector dominates the final context vector ($z$).
* This results in a new representation that "knows" about its neighbors.